<a href="https://colab.research.google.com/github/dataprogpy/code-samples/blob/main/starter_files/03_introduction_to_polars.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Data Wrangling with Polars

**Overall Goal:** Equip students with the foundational skills to load, inspect, clean, transform, and aggregate data using Polars, understanding its core API concepts and how these operations fit into a typical data analysis workflow.

## Setup

First, let's import the Polars library. The standard alias is `pl`.

In [ ]:
import polars as pl
import polars.selectors as cs # For using column selectors like cs.numeric()

# Print Polars version for reference
print(f"Polars version: {pl.__version__}")

---
## 1: Introduction to Data Wrangling & Polars

This section introduces the foundational concepts of data wrangling and the Polars library, a powerful tool for this purpose. Effective data wrangling is a critical precursor to sound business analysis and data-driven decision-making.

### 1.1. The Imperative of Data Wrangling in Business Analytics

In the context of business analytics, **data wrangling** refers to the process of transforming and mapping raw data from its original state into a clean, structured, and validated format suitable for analysis. This process is also commonly referred to as data munging or data tidying.

* **The "80/20 Rule" in Data Preparation:** It is widely recognized within the data science community that data preparation activities, including wrangling, can consume as much as 80% of the total time and effort in an analytical project. The remaining 20% is typically spent on performing the actual analysis and generating insights. This highlights the substantial resources dedicated to ensuring data quality and usability.
* **Data Sources and the Need for Repurposing:**
    Data utilized for business analytics is often sourced from systems not initially designed for that specific analytical objective. These sources can include:
    * **Online Transaction Processing (OLTP) Systems:** Operational systems (e.g., sales, CRM, ERP) optimized for transaction recording, not complex queries.
    * **Online Analytical Processing (OLAP) Systems:** Designed for analysis, but specific questions may require further manipulation or integration.
    * **Data Lakes:** Repositories of vast raw data needing significant wrangling.
    * **Governance and Regulatory Data Archival Systems:** Data structured for compliance, often requiring transformation for analytics.

    Because these diverse systems serve different primary functions, the data they produce rarely aligns perfectly with the schema (structure) required for a specific analytical investigation. Consequently, data wrangling becomes an essential intermediary step.

* **Illustrative Data Challenges (Preview for our datasets `customers.csv` and `orders.csv`):
    * Inconsistent date formats.
    * Missing values in critical fields.
    * Outliers or erroneous data entries.

### 1.2. Introduction to Polars: A High-Performance DataFrame Library

* **Overview:** Polars is a modern DataFrame library implemented in Rust, designed for high performance and an expressive API in Python.
* **Advantages:** Speed, expressive syntax, modern capabilities.

### 1.3. Core Polars Concepts: A High-Level View

* **Expressions:** Blueprints for operations (`pl.col()`, `pl.lit()`).
* **Execution Strategies (Lazy vs. Eager):** Polars can optimize a plan of operations (lazy) or execute immediately (eager).

### 1.4. Setup: Importing the Polars Library

We've already done this in the setup cell above: `import polars as pl`.

---
## 2: Getting Data In - Data IO

This section focuses on loading data into Polars DataFrames from common file formats.

**Note for Google Colab users:** You will need to upload `customers.csv` and `orders.csv` to your Colab environment. You can do this by clicking the folder icon on the left sidebar, then the "Upload to session storage" icon.

In [ ]:
# TODO: If using Google Colab, uncomment and run these lines to upload files.
from google.colab import files
print("Please upload customers.csv and orders.csv")
uploaded = files.upload()
#
for fn in uploaded.keys():
  print('User uploaded file "{name}" with length {length} bytes'.format(
      name=fn, length=len(uploaded[fn])))

### 2.1 Reading Comma-Separated Values (CSV) Files

We use `pl.read_csv()` to load data from CSV files.

In [ ]:
# Load the customers dataset
customers_df = pl.read_csv('customers.csv')
print("customers.csv loaded successfully.")
# Display a sample to verify
print("Sample of customers_df:")
customers_df.head(3)

In [ ]:
# Load the orders dataset
orders_df = pl.read_csv('orders.csv')
print("\norders.csv loaded successfully.")
# Display a sample to verify
print("Sample of orders_df:")
orders_df.head(3)


#### Specifying or Overriding Data Types (Schema) with `dtypes`

You can explicitly define column data types during import for correctness, performance, or memory efficiency.

In [ ]:
# Define the desired schema for selected columns in customers.csv
customer_schema = {
    'customer_id': pl.Int32,
    'name': pl.Utf8,
    'registration_date': pl.Utf8, # Read as string; parsing will be handled later
    'city': pl.Categorical,
    'age': pl.Int64 # Allow for nulls, but keep as Int64 for now
}

In [ ]:
customers_custom_schema_df = pl.read_csv('customers.csv', schema_overrides=customer_schema)
print("\ncustomers.csv loaded with custom schema successfully.")
print("Schema of customers_custom_schema_df:")
customers_custom_schema_df.schema

For subsequent sections, we'll primarily use the initially loaded customers_df and orders_df (without the custom schema for customers_df initially) to demonstrate transformations like type casting. However, let's rename the original date columns to clarify they are strings before parsing in later sections.

In [ ]:
customers_df = customers_df.rename({"registration_date":"registration_date_str"})
orders_df = orders_df.rename({"order_date": "order_date_str"})

### 2.2 Reading CSV Data from a URL

Code example from [Polars User Guide](https://docs.pola.rs/user-guide/expressions/window-functions/)

In [ ]:
types = (
    "Grass Water Fire Normal Ground Electric Psychic Fighting Bug Steel "
    "Flying Dragon Dark Ghost Poison Rock Ice Fairy".split()
)
# Create an enumeration type
type_enum = pl.Enum(types)
# then let's load some csv data with information about pokemon
pokemon = pl.read_csv(
    "https://gist.githubusercontent.com/ritchie46/cac6b337ea52281aa23c049250a4ff03/raw/89a957ff3919d90e6ef2d34235e6bf22304f3366/pokemon.csv",
).cast({"Type 1": type_enum, "Type 2": type_enum})
print(pokemon.head())


### 2.3. Reading JSON Files

Polars can read JSON files, typically those structured as a list of records.

First, let's create a sample JSON file named `product_inventory.json` in our working directory. You can do this by running the code cell below, which writes the JSON string to a file.

In [ ]:
product_inventory_json_content = """
[
  {\"product_id\": \"P1001\", \"product_name\": \"Laptop X1\", \"category\": \"Electronics\", \"stock_level\": 150, \"reorder_point\": 50},
  {\"product_id\": \"P1002\", \"product_name\": \"Wireless Mouse\", \"category\": \"Electronics\", \"stock_level\": 300, \"reorder_point\": 100},
  {\"product_id\": \"P1003\", \"product_name\": \"Office Chair Pro\", \"category\": \"Furniture\", \"stock_level\": 85, \"reorder_point\": 30},
  {\"product_id\": \"P1004\", \"product_name\": \"Standing Desk\", \"category\": \"Furniture\", \"stock_level\": 60, \"reorder_point\": 20}
]
"""

with open('product_inventory.json', 'w') as f:
    f.write(product_inventory_json_content)

print("'product_inventory.json' created successfully.")

In [ ]:
inventory_df = pl.read_json('product_inventory.json')
print("\nProduct inventory data loaded successfully from JSON.")
inventory_df.head()

---
## 3: First Look - Initial Data Exploration & Understanding

This section covers fundamental techniques to inspect DataFrames.

**Make sure `customers_df` and `orders_df` are loaded from section 2 before proceeding.**

### 3.1. Understanding DataFrame Structure: Dimensions (`.shape`)

In [ ]:
customer_dimensions = customers_df.shape
print(f"Customers DataFrame - Rows: {customer_dimensions[0]}, Columns: {customer_dimensions[1]}")

order_dimensions = orders_df.shape
print(f"Orders DataFrame - Rows: {order_dimensions[0]}, Columns: {order_dimensions[1]}")

### 3.2. Inspecting Data Content: First and Last Rows (`.head()`, `.tail()`)

In [ ]:
print("First 3 rows of customers_df:")
customers_df.head(3)

In [ ]:
# Your turn: Display the last 4 rows of the orders_df
print("\nLast 4 rows of orders_df:")
# TODO: Write your code here


### 3.3. Examining Data Types and Schema (`.dtypes`, `.schema`)

In [ ]:
print("Data types for customers_df:")
print(customers_df.dtypes)

print("\nSchema for customers_df:")
print(customers_df.schema)

### 3.4. Obtaining Descriptive Statistics and a Quick Overview (`.describe()`, `.glimpse()`)

In [ ]:
print("Descriptive statistics for orders_df:")
orders_df.describe() # Typically shows numerical columns

In [ ]:
print("\nGlimpse of customers_df:")
customers_df.glimpse()

### 3.5. Identifying and Quantifying Missing Values using Expressions

In [ ]:
customers_df.select(
    pl.all().is_null().sum()
)

In [ ]:
# Your turn: Count missing values in each column of orders_df using the expression API
print("\nMissing values per column in orders_df (via expression):")
# TODO: Write your code here


### 3.6. Analyzing Unique Values in Columns and Creating Enums using Expressions

In [ ]:
print("Number of unique cities in customers_df (via expression):")
customers_df.select(
    pl.col('city').n_unique().alias("unique_city_count")
)

In [ ]:
print("\nUnique cities in customers_df (via expression):")
customers_df.select(
    pl.col('city').unique().sort()
)

---
## 4: Core Polars Concepts - The Building Blocks

This section delves into DataFrames, Series, and Expressions.

### 4.1. Polars DataFrame Revisited
A 2D, in-memory, tabular data structure.

In [ ]:
data_dict = {
    'StudentID': [1001, 1002, 1003],
    'Course': ['Finance', 'Marketing', 'Finance'],
    'MidtermScore': [85, 92, 78]
}
scores_df = pl.DataFrame(data_dict)
print("DataFrame created from scratch:")
scores_df

### 4.2. Polars Series
A 1D array-like object representing a single column.

In [ ]:
city_series = customers_df.get_column('city')
print("'city' column extracted as a Series:")
city_series.head()

In [ ]:
print(f"Type of city_series: {type(city_series)}")
print(f"Name of city_series: {city_series.name}")
print(f"DataType of city_series: {city_series.dtype}")

### 4.3. Polars Expressions: The Core Engine
Blueprints for operations, evaluated in contexts.

#### Referring to Columns: `pl.col()`
#### Literal Values: `pl.lit()`
#### Basic Operations with Expressions

In [ ]:
original_price_expr = pl.col("unit_price")
discount_rate_lit_expr = pl.lit(0.05) # 5% discount
discounted_price_expr = original_price_expr * (pl.lit(1) - discount_rate_lit_expr)
print(f"Expression for discounted price: {discounted_price_expr}")

is_older_than_30_expr = pl.col("age") > pl.lit(30)
print(f"Expression for 'age > 30': {is_older_than_30_expr}")

#### Expressions are Executed in Contexts (Preview for sections 5 & 6)
Different contexts use expressions for different purposes.

In [ ]:
print("Preview: Expression in `select` context")
customer_status_df = customers_df.select([
    pl.col("name"),
    pl.col("age"),
    (pl.col("age") < pl.lit(18)).alias("is_minor")
])
customer_status_df.head()

In [ ]:
print("\nPreview: Expression in `with_columns` context")
orders_with_total = orders_df.with_columns(
    (pl.col("quantity") * pl.col("unit_price")).alias("total_price")
)
orders_with_total.head()

In [ ]:
print("\nPreview: Expression in `filter` context")
# Using the raw 'age' column for this example. Note the outlier.
adult_customers_df = customers_df.filter(
    pl.col("age") > pl.lit(30)
)
adult_customers_df.select(["name", "age"])


In [ ]:
print("\nPreview: Expression in `group_by().agg()` context")
sales_by_category = orders_df.group_by("product_category").agg([
    pl.sum("quantity").alias("total_quantity_sold"),
    pl.mean("unit_price").alias("average_unit_price")
])
sales_by_category

---
## 5: Transforming Data - Selection, Filtering, and Modification

This section covers primary operations for transforming data.

**Ensure `customers_df` and `orders_df` are loaded and column names `registration_date_str` and `order_date_str` are set as per section 2 modifications.**

### 5.1. Column Selection and Manipulation with `select`

In [ ]:
customer_locations_df = customers_df.select([
    pl.col("name"),
    pl.col("city")
])
print("Selected customer names and cities:")
print(customer_locations_df.head())

renamed_customers_df = customers_df.select([
    pl.col("customer_id").alias("ID"),
    pl.col("name").alias("Customer Name")
])
print("\nCustomers DataFrame with renamed columns:")
print(renamed_customers_df.head())

#### Using Polars Selectors (`cs`)

In [ ]:
string_cols_orders_df = orders_df.select(cs.string())
print("String columns from orders_df:")
print(string_cols_orders_df.head())

# Your turn: Select all numeric columns from orders_df
print("\nNumeric columns from orders_df:")
# TODO: Write your code here
numeric_cols_orders_df = orders_df.select(cs.numeric())
print(numeric_cols_orders_df.head())

### 5.2. Adding or Modifying Columns with `with_columns`

In [ ]:
orders_enhanced_df = orders_df.with_columns([
    (pl.col("quantity") * pl.col("unit_price")).alias("total_price"),
    pl.col("discount_applied").is_not_null().alias("has_discount")
])
print("Orders DataFrame with 'total_price' and 'has_discount' columns:")
print(orders_enhanced_df.head())

In [ ]:
# Your turn: In customers_df, create a new column 'name_length' that contains the length of the customer's name.
customers_with_name_length_df = customers_df.with_columns(
    # TODO: Write your expression here
    pl.col("name").str.len_chars().alias("name_length")
)
print("\nCustomers DataFrame with name_length:")
print(customers_with_name_length_df.select(["name", "name_length"]).head())


### 5.3. Filtering Rows with `filter`

In [ ]:
electronics_orders_df = orders_df.filter(
    pl.col("product_category") == pl.lit("Electronics")
)
print("Electronics orders:")
print(electronics_orders_df.head())

In [ ]:
# Note: The 'age' column in customers.csv has an outlier (3000) and missing values.
# For realistic filtering, this should be handled.
# Let's create a cleaned version for this filtering example.
customers_cleaned_age_df = customers_df.with_columns(
    pl.when(pl.col("age") > 100).then(None).otherwise(pl.col("age")).cast(pl.Int64, strict=False).alias("age_cleaned")
)
older_customers_df = customers_cleaned_age_df.filter(
    pl.col("age_cleaned") > pl.lit(50)
)
print("\nCustomers older than 50 (age cleaned):")
print(older_customers_df.select(["name", "age_cleaned", "city"]))

In [ ]:
# Your turn: Filter orders_df for products in the 'Home Goods' category OR where 'quantity' is greater than 2.
print("\nHome Goods orders OR quantity > 2:")
# TODO: Write your code here
filtered_orders_complex = orders_df.filter(
    (pl.col("product_category") == pl.lit("Home Goods")) | (pl.col("quantity") > pl.lit(2))
)
print(filtered_orders_complex)

### 5.4. Data Type Conversion (Casting)
Using `pl.col().cast(DataType)` or specific conversion methods like `str.to_datetime()`.

In [ ]:
# Casting 'city' to Categorical in customers_df
customers_city_categorical_df = customers_df.with_columns(
    pl.col("city").cast(pl.Categorical).alias("city_categorical")
)
print("Customers DataFrame with city as Categorical:")
print(customers_city_categorical_df.select(["name", "city_categorical"]).head())
print(f"Data type of 'city_categorical': {customers_city_categorical_df.get_column('city_categorical').dtype}")

In [ ]:
# Parsing 'order_date_str' in orders_df to Datetime
orders_parsed_dates_df = orders_df.with_columns(
    pl.col("order_date_str").str.to_datetime(format="%Y-%m-%d %H:%M:%S", strict=False).alias("order_datetime")
)
print("\nOrders DataFrame with parsed order_datetime:")
print(orders_parsed_dates_df.select(["order_id", "order_datetime"]).head())
print(f"Data type of 'order_datetime': {orders_parsed_dates_df.get_column('order_datetime').dtype}")

In [ ]:
# Parsing 'registration_date_str' in customers_df (handles mixed formats and nulls)
customers_parsed_reg_dates_df = customers_df.with_columns(
    pl.coalesce([
        pl.col("registration_date_str").str.to_date(format="%Y-%m-%d", strict=False),
        pl.col("registration_date_str").str.to_date(format="%m/%d/%Y", strict=False)
    ]).alias("registration_date")
)
print("\nCustomers DataFrame with parsed registration_date:")
print(customers_parsed_reg_dates_df.select(["name", "registration_date_str", "registration_date"]))
print(f"Data type of 'registration_date': {customers_parsed_reg_dates_df.get_column('registration_date').dtype}")

### 5.5. Handling Missing Values
Using `fill_null()` or `drop_nulls()`.

In [ ]:

customers_cleaned_age_df = customers_df.with_columns(
  pl.when(pl.col("age") > 100).then(None).otherwise(pl.col("age")).cast(pl.Float64, strict=False).alias("age_cleaned")
)

# Calculate mean age for filling, excluding extreme outliers for a more sensible mean
# Note: .item() extracts the single value from a 1x1 DataFrame/Series
mean_age_val = customers_cleaned_age_df.select(pl.col("age_cleaned").mean()).item()

customers_filled_age_df = customers_cleaned_age_df.with_columns(
    pl.col("age_cleaned").fill_null(mean_age_val).alias("age_filled_with_mean")
)
print(f"Customers DataFrame with 'age_cleaned' nulls filled with mean ({mean_age_val:.2f}):")
print(customers_filled_age_df.filter(customers_df["age"].is_null()).select(["name", "age", "age_cleaned", "age_filled_with_mean"]))

In [ ]:
# Your turn: In orders_df, fill missing 'discount_applied' with 0.0 and missing 'quantity' with 1.
orders_filled_df = orders_df.with_columns([
    # TODO: Fill 'discount_applied' nulls with 0.0
    pl.col("discount_applied").fill_null(0.0).alias("discount_applied_filled"),
    # TODO: Fill 'quantity' nulls with 1 (assuming a missing quantity implies at least one item)
    pl.col("quantity").fill_null(1).alias("quantity_filled")
])
print("\nOrders DataFrame with nulls filled for discount and quantity:")
print(orders_filled_df.filter(pl.col("discount_applied").is_null() | pl.col("quantity").is_null()).select(["order_id", "discount_applied", "discount_applied_filled", "quantity", "quantity_filled"]).head())
# Verify no nulls remain in these specific columns
print("\nNull counts after filling:")
print(orders_filled_df.select(["discount_applied_filled", "quantity_filled"]).is_null().sum())

In [ ]:
# Dropping rows with nulls
# Drop rows from orders_df where 'quantity' is null (using original orders_df for this example)
orders_dropped_null_quantity_df = orders_df.drop_nulls(subset=["quantity"])
print(f"Original orders count: {orders_df.height}, After dropping null quantity: {orders_dropped_null_quantity_df.height}")

### 5.6. Sorting Data with `sort()`

In [ ]:
sorted_customers_by_age_df = customers_cleaned_age_df.sort("age_cleaned", descending=True)
print("Customers sorted by cleaned age (descending):")
print(sorted_customers_by_age_df.select(["name", "age_cleaned", "city"]).head())

---
## 6: Aggregating & Reshaping Data - Summarizing for Insights

This section covers grouping data and applying aggregation functions.

### 6.1. Group By and Aggregations (`.group_by().agg()`)
The "split-apply-combine" strategy.

In [ ]:
category_summary_df = orders_df.group_by("product_category").agg([
    pl.sum("quantity").alias("total_quantity_sold"),
    pl.mean("unit_price").alias("average_unit_price"),
    pl.col("order_id").count().alias("number_of_orders"),
    pl.n_unique("customer_id").alias("unique_customers_in_category")
]).sort("total_quantity_sold", descending=True)

print("Sales summary by product category:")
print(category_summary_df)

In [ ]:
# Your turn: Using customers_cleaned_age_df, group by 'city' and find:
# 1. The number of customers in each city.
# 2. The average cleaned age of customers in each city.
# Sort the results by the number of customers in descending order.
print("\nCustomer demographics by city:")
# TODO: Write your code here


### 6.2. Pivoting Data with `df.pivot()`

In [ ]:
# Ensure order_datetime is parsed for year extraction
if 'order_datetime' not in orders_df.columns:
    current_orders_df = orders_df.with_columns(
          pl.col("order_date_str").str.to_datetime(format="%Y-%m-%d %H:%M:%S", strict=False).alias("order_datetime")
    )
else:
    current_orders_df = orders_df

orders_with_year_df = current_orders_df.with_columns(
    pl.col("order_datetime").dt.year().alias("order_year")
)

yearly_category_sales = orders_with_year_df.group_by(["order_year", "product_category"]).agg(
    pl.sum("quantity").alias("total_quantity")
).sort(["order_year", "product_category"])

print("Yearly sales by category (long format):")
print(yearly_category_sales.head())

pivoted_yearly_sales_df = yearly_category_sales.pivot(
    values="total_quantity",
    index="order_year",
    columns="product_category"
).fill_null(0)

print("\nPivoted yearly sales (product categories as columns):")
print(pivoted_yearly_sales_df)

### 6.3. Window Functions
Window functions perform calculations across a set of table rows related to the current row. Examples include running totals, moving averages, and ranking within groups.

In [ ]:
# Ensure order_datetime is parsed and quantity is numeric
if 'order_datetime' not in orders_df.columns:
    current_orders_df_window = orders_df.with_columns([
          pl.col("order_date_str").str.to_datetime(format="%Y-%m-%d %H:%M:%S", strict=False).alias("order_datetime"),
          pl.col("quantity").fill_null(0) # Ensure quantity is filled for cumsum
    ])
else:
    current_orders_df_window = orders_df.with_columns(pl.col("quantity").fill_null(0))

orders_sorted_for_window = current_orders_df_window.sort(["product_category", "order_datetime"])

orders_with_running_total = orders_sorted_for_window.with_columns(
    pl.col("quantity").cumsum().over("product_category").alias("running_total_quantity_in_category")
)
print("Orders with running total quantity within category (illustrative):")
print(orders_with_running_total.filter(pl.col("product_category") == "Books").select([
    "order_datetime", "product_category", "quantity", "running_total_quantity_in_category"
]).tail(5))

---
## 7: Stitching and Saving Data

This section covers joining related datasets and persisting processed data.

### 7.1. Stitching Data: Joining DataFrames with `.join()`

In [ ]:
merged_inner_df = orders_df.join(
    customers_df, # The DataFrame to join with
    on="customer_id", # The common column to join on
    how="inner"       # The type of join
)
print("Inner Join - Orders with Customer Details (first 5 rows):")
print(merged_inner_df.select([
    "order_id", "customer_id", "name", "city", "product_category", "unit_price"
]).head())
print(f"Shape of inner joined df: {merged_inner_df.shape}")

# Left join: All customers, with their order details if available
customer_orders_left_df = customers_df.join(
    orders_df,
    on="customer_id",
    how="left"
)
print("\nLeft Join - All Customers with Their Orders (if any) (first 10 rows):")
print(customer_orders_left_df.select([
    "customer_id", "name", "city", "order_id", "product_category"
]).sort("customer_id").head(10))
# Customers with no orders will have null for order_id and other order-specific columns
print("\nCustomers with potentially no orders (order_id is null from left join):")
print(customer_orders_left_df.filter(pl.col("order_id").is_null()).select(["customer_id", "name", "city"]))

### 7.2. Saving DataFrames to Files

In [ ]:
output_df_for_csv = merged_inner_df.select([
    "order_id", "customer_id", "name", "city", "product_category", "quantity", "unit_price", "age_cleaned", "registration_date_str"
]).sort("order_id") # Select a few more columns for richness

In [ ]:
output_df_for_csv.write_csv("enriched_orders_report.csv")
print("\nSuccessfully saved 'enriched_orders_report.csv'")

In [ ]:
output_df_for_csv.head(10).write_json("enriched_orders_sample.json", row_oriented=True, pretty=True)
print("Successfully saved 'enriched_orders_sample.json' (first 10 rows, row-oriented, pretty)")

### 7.3. Workflow and Execution Notes (Briefly)

In [ ]:
# Method Chaining Example
# Assume customers_parsed_reg_dates_df and orders_parsed_dates_df were created in section 5
# For simplicity, let's ensure our main dfs have parsed dates if we use them.
customers_df_final = customers_df.with_columns([
    pl.coalesce([
        pl.col("registration_date_str").str.to_date(format="%Y-%m-%d", strict=False),
        pl.col("registration_date_str").str.to_date(format="%m/%d/%Y", strict=False)
    ]).alias("registration_date"),
    # Ensure age_cleaned is present
    pl.when(pl.col("age") > 100).then(None).otherwise(pl.col("age")).cast(pl.Int64, strict=False).alias("age_cleaned")
])

orders_df_final = orders_df.with_columns([
    pl.col("order_date_str").str.to_datetime(format="%Y-%m-%d %H:%M:%S", strict=False).alias("order_datetime"),
    pl.col("quantity").fill_null(0), # Fill null quantities for calculation
    pl.col("unit_price").fill_null(0.0) # Fill null prices for calculation
])

# Chained operation: Find total sales amount for customers in 'New York' for 'Electronics' orders
ny_electronics_sales = (
    orders_df_final
    .join(customers_df_final, on="customer_id", how="inner")
    .filter(
        (pl.col("city") == pl.lit("New York")) &
        (pl.col("product_category") == pl.lit("Electronics"))
    )
    .with_columns(
        (pl.col("quantity") * pl.col("unit_price")).alias("sale_amount")
    )
    .group_by("name") # Group by customer name
    .agg(
        pl.sum("sale_amount").alias("total_spent_on_electronics")
    )
    .sort("total_spent_on_electronics", descending=True)
)
print("\nChained operations - NY Electronics Sales by Customer:")
print(ny_electronics_sales)

In [ ]:
# Lazy Execution and .collect() Revisited
lazy_result_plan = (
    orders_df.lazy() # Start a lazy query
    .filter(pl.col("unit_price") > pl.lit(500))
    .group_by("product_category")
    .agg(pl.sum("quantity").alias("total_high_value_quantity"))
)
print("\nLazy query plan constructed (no execution yet):")
print(lazy_result_plan)

print("\nExecuting lazy query with .collect():")
final_high_value_summary_df = lazy_result_plan.collect()
print(final_high_value_summary_df)

---
## 8. Active Learning Exercises

Complete the following tasks to practice the concepts learned in this lesson. Use the `customers_df` and `orders_df` DataFrames.

**Instructions:**
1. Ensure `customers.csv` and `orders.csv` are loaded into `customers_df` and `orders_df` respectively. You may need to re-run cells from section 2 if you've restarted your notebook.
2.  Remember to handle data type conversions (especially for dates) and missing values where appropriate for accurate analysis.
3.  Write your Polars code in the cells provided below each task.

### Task 1: Customer Age Analysis

a. From `customers_df`, identify and count how many customers have an `age` recorded as the outlier value (3000).
b. Create a new DataFrame `customers_age_cleaned_df` where ages greater than 100 are replaced with `None` (null). Then, fill any remaining null ages (including the newly created ones) with the median age of the *cleaned* distribution.
c. What is the average `age` (from your cleaned and filled data) of customers registered in the year 2023? (Hint: You'll need to parse `registration_date_str` first).

In [ ]:
# Task 1a: Count outlier ages
print("Task 1a:")
# Your code here


In [ ]:
# Task 1b: Clean and fill ages
print("\nTask 1b:")
# Your code here


In [ ]:
# Task 1c: Average age of customers registered in 2023
print("\nTask 1c:")
# Your code here


### Task 2: Order Insights

a. In `orders_df`, calculate the actual discount amount for each order (unit_price * quantity * discount_applied). Store this in a new column called `discount_value`. Handle cases where `discount_applied` or `quantity` might be null (assume 0 for discount_applied and 1 for quantity if null for this calculation).
b. Which `product_category` has the highest total `discount_value` given to customers?
c. For orders placed in Q1 (January, February, March) of any year, what is the count of unique customers?

In [ ]:
# Task 2a: Calculate discount_value
print("Task 2a:")
# Your code here


In [ ]:
# Task 2b: Product category with highest total discount_value
print("\nTask 2b:")
# Your code here


In [ ]:
# Task 2c: Unique customers in Q1 orders
print("\nTask 2c:")
# Your code here


### Task 3: Combined Analysis & Output

a. Join `orders_df` (after any necessary cleaning/preparation from Task 2, e.g., having `discount_value`) with `customers_df` (using cleaned age data from Task 1, `customers_age_cleaned_df`).
b. From the joined DataFrame, find the top 3 cities by total sales revenue (quantity * unit_price - discount_value). For simplicity, if `discount_value` isn't readily available from Task 2a, you can calculate total revenue as (quantity * unit_price) and ignore discounts for this specific sub-task, but note this assumption.
c. Save the resulting summary (city and total sales revenue) from Task 3b to a CSV file named `city_revenue_report.csv`.

In [ ]:
# Task 3a: Join cleaned orders and customers data
print("Task 3a:")
# Your code here


In [ ]:
# Task 3b: Top 3 cities by total sales revenue
print("\nTask 3b:")
# Your code here


In [ ]:
# Task 3c: Save city revenue report
print("\nTask 3c:")
# Your code here


---
End of Notebook. Remember to submit your completed tasks.

## Polars Documentation


### User Guide

- [Expressions](https://docs.pola.rs/user-guide/expressions/)
- [Basic Operations](https://docs.pola.rs/user-guide/expressions/basic-operations/)
- [Strings](https://docs.pola.rs/user-guide/expressions/strings/)
- [Time Series](https://docs.pola.rs/user-guide/transformations/time-series/parsing/)
- [Casting](https://docs.pola.rs/user-guide/expressions/casting/)
- [Missing Data](https://docs.pola.rs/user-guide/expressions/missing-data/)
- [Enums & Categorical Data](https://docs.pola.rs/user-guide/expressions/categorical-data-and-enums/)
- [Aggregation](https://docs.pola.rs/user-guide/expressions/aggregation/)
- [Concatenation](https://docs.pola.rs/user-guide/transformations/concatenation/)
- [Join](https://docs.pola.rs/user-guide/transformations/joins/)

### API Reference

- [API Reference Home](https://docs.pola.rs/api/python/stable/reference/index.html)
- [DataFrame](https://docs.pola.rs/api/python/stable/reference/dataframe/index.html)
- [Series](https://docs.pola.rs/api/python/stable/reference/series/index.html)
- [Expressions](https://docs.pola.rs/api/python/stable/reference/expressions/index.html )
- [LazyFrame](https://docs.pola.rs/api/python/stable/reference/lazyframe/index.html )
- [Selectors](https://docs.pola.rs/api/python/stable/reference/selectors.html )
- [IO](https://docs.pola.rs/api/python/stable/reference/io.html)

